In [ ]:
import os
import chromadb
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFDirectoryLoader
from dotenv import load_dotenv

In [3]:
# Load environment variables
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [4]:
gemini_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/embedding-001",
)

In [5]:
from langchain_chroma import Chroma

In [6]:

loader = PyPDFDirectoryLoader("../product_db")
data = loader.load()

In [7]:
data

[Document(metadata={'producer': 'Skia/PDF m139 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'intellishelf_guidence', 'source': '../product_db/intellishelf_guidence.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='IntelliShelf™  –\u2009Store  Policies  &  Customer  Assurance  Guide   *(Last  updated\u200921\u2009June\u20092025)*       1.\u2009About  IntelliShelf   IntelliShelf  is  an  online  boutique  dedicated  to  world ‑ class,  stylish,  and  beautifully  crafted  \nmusical\n \ninstruments\n.\n \nOur\n \ncurated\n \ncatalog\n \nfocuses\n \non\n \nfive\n \nsignature\n \ncategories:\n \nviolin,\n \ndrum,\n \ntabla,\n \nguitar,\n \nand\n \nflute\n.\n \nEvery\n \nitem\n \nis\n \ninspected\n \nby\n \nboth\n \nhuman\n \nexperts\n \nand\n \nour\n \nproprietary\n \nAI\n \nvision\n \nsystem\n \nbefore\n \nshipment.\n     2.\u2009Return  Policy    Scenario  Return  Window  How  It  Works  \nChange  of  mind  /  wrong  item  (undamaged  product)

In [8]:

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

In [9]:
text_chunks = text_splitter.split_documents(data)

In [10]:
text_chunks[0]
     

Document(metadata={'producer': 'Skia/PDF m139 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'intellishelf_guidence', 'source': '../product_db/intellishelf_guidence.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='IntelliShelf™  –\u2009Store  Policies  &  Customer  Assurance  Guide   *(Last  updated\u200921\u2009June\u20092025)*       1.\u2009About  IntelliShelf   IntelliShelf  is  an  online  boutique  dedicated  to  world ‑ class,  stylish,  and  beautifully  crafted  \nmusical\n \ninstruments\n.\n \nOur\n \ncurated\n \ncatalog\n \nfocuses\n \non\n \nfive\n \nsignature\n \ncategories:\n \nviolin,\n \ndrum,\n \ntabla,\n \nguitar,\n \nand\n \nflute\n.\n \nEvery\n \nitem\n \nis\n \ninspected\n \nby\n \nboth\n \nhuman\n \nexperts\n \nand\n \nour\n \nproprietary\n \nAI\n \nvision\n \nsystem')

In [11]:
persist_directory = '../chromadb'

In [12]:
vectorstore = Chroma.from_documents(
    documents=text_chunks,
    embedding=gemini_embeddings,
    persist_directory=persist_directory
)

In [17]:
from langchain_chroma import Chroma
from langchain_community.vectorstores import Chroma

In [12]:

# Now we can load the persisted database from disk, and use it as normal.
vectorstore = Chroma(persist_directory=persist_directory,
                  embedding_function=gemini_embeddings)

In [13]:
retriever = vectorstore.as_retriever()
retriever

VectorStoreRetriever(tags=['Chroma', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x11caa5c60>, search_kwargs={})

In [14]:
docs = retriever.get_relevant_documents("what is return policy of intellishelf?")

/var/folders/v3/hj2d2g150gld7k2cm6q9d9gh0000gn/T/ipykernel_77442/3754919621.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents("what is return policy of intellishelf?")


In [15]:
docs

[Document(id='86a5fda7-a715-4c37-9827-c503d6a5a216', metadata={'creationdate': '', 'total_pages': 4, 'source': '../product_db/intellishelf_guidence.pdf', 'producer': 'Skia/PDF m139 Google Docs Renderer', 'page': 0, 'page_label': '1', 'title': 'intellishelf_guidence', 'creator': 'PyPDF'}, page_content='•  Initiate  a  return  request  in     your  IntelliShelf  account.  •  Items  must  be  unused,  in  original  packaging,  with  all  accessories.  •  A  prepaid  return  label  will  be  issued.  •  Refund  or  store  credit  processed  within  3\u2009business\u2009days  of  warehouse  receipt.  \nProduct  delivered  damaged  or  defective  \n48\u2009hours  from  delivery  timestamp'),
 Document(id='5cbb7c73-4ae9-4a67-94ef-06db8381c603', metadata={'title': 'intellishelf_guidence', 'page': 2, 'page_label': '3', 'creationdate': '', 'creator': 'PyPDF', 'source': '../product_db/intellishelf_guidence.pdf', 'total_pages': 4, 'producer': 'Skia/PDF m139 Google Docs Renderer'}, page_content='fo

In [16]:
system_prompt = (
    "You are an assistant for question answering tasks. "
    "Use the following pieces of retrieved context to answer the question "
    "If you don't know the answer, say that you don't know."
    "Use three sentences maximum and keep the answer concise."
    "\n\n"
    "{context}"
)

In [17]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [18]:
model = ChatGoogleGenerativeAI(model = "gemini-1.5-flash", convert_system_message_to_human=True)

In [19]:
from langchain.chains.combine_documents import create_stuff_documents_chain

question_answering_chain = create_stuff_documents_chain(model, chat_prompt)

In [20]:
from langchain.chains import create_retrieval_chain

rag_chain = create_retrieval_chain(retriever, question_answering_chain)

In [21]:
rag_chain.invoke({"input": "what is return policy of intellishelf?"})["answer"]

/Users/yash/Library/IntelliShelf/IntelliShelf/lib/python3.10/site-packages/langchain_google_genai/chat_models.py:424: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


"IntelliShelf's return policy allows for returns within 7 days for change of mind or wrong item, and within 48 hours for damaged or defective products.  Items must be unused and in original packaging.  A prepaid return label will be provided, and refunds are processed within 3 business days of warehouse receipt."

In [22]:
rag_chain.invoke({"input": "what is warranty of my product?"})["answer"]

/Users/yash/Library/IntelliShelf/IntelliShelf/lib/python3.10/site-packages/langchain_google_genai/chat_models.py:424: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


'The product has a 1-year limited warranty from the purchase date.  This covers manufacturing faults like cracked solder joints or warped necks (excluding damage from humidity, negligence, or misuse).  Damage not covered includes accidental breakage and normal wear and tear.'

In [23]:
rag_chain.invoke({"input": "how many categories do you sell??"})["answer"]

/Users/yash/Library/IntelliShelf/IntelliShelf/lib/python3.10/site-packages/langchain_google_genai/chat_models.py:424: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


"IntelliShelf sells five categories of musical instruments.  These are violin, drum, tabla, guitar, and flute.  The company's curated catalog focuses on these instruments."